In [1]:
csv_text = """route_id,route_name,terminal,scheduled_arrival_time
RT-001,Whitefield - Majestic,Majestic Bus Stand,08:15:00
RT-002,Electronic City - KR Puram,KR Puram Terminal,08:20:00
RT-003,Hebbal - Jayanagar,Jayanagar Terminal,08:10:00
RT-004,Yeshwanthpur - Silk Board,Silk Board Terminal,08:25:00
RT-005,Banashankari - Marathahalli,Marathahalli Terminal,08:05:00
"""
with open("route_schedule.csv", "w") as f:
    f.write(csv_text)

In [3]:
import csv, json
from confluent_kafka import Consumer, Producer

BOOTSTRAP = "localhost:9092,localhost:9094,localhost:9096"

def load_route_table(path):        # this dict IS the KTable
    with open(path, newline="") as f:
        return {row["route_id"]: row for row in csv.DictReader(f)}

route_table = load_route_table("route_schedule.csv")
print(f"Loaded KTable with {len(route_table)} routes")

consumer = Consumer({"bootstrap.servers": BOOTSTRAP, "group.id": "urbanpulse-route-enrichment",
                      "auto.offset.reset": "latest"})
consumer.subscribe(["urbanpulse.bus_gps"])
producer = Producer({"bootstrap.servers": BOOTSTRAP})

while True:
    msg = consumer.poll(0.5)
    if msg is None or msg.error():
        continue
    gps = json.loads(msg.value())
    route = route_table.get(gps["route_id"])        # <-- the KStream-KTable join
    enriched = {**gps,
                "scheduled_arrival_time": route["scheduled_arrival_time"] if route else None,
                "route_name": route["route_name"] if route else "UNKNOWN",
                "terminal": route["terminal"] if route else "UNKNOWN"}
    producer.produce("urbanpulse.bus_gps_enriched", key=gps["route_id"].encode(),
                      value=json.dumps(enriched).encode())
    producer.poll(0)

Loaded KTable with 5 routes


KeyboardInterrupt: 